# Log-odds, and a cut at one half

A logistic model writes a probability through the sigmoid

$$
\sigma(z) = \dfrac{1}{1+e^{-z}}.
$$

Start with a sketch that has a closed form. Five customers on long contracts, 1 of whom leaves, and five on short contracts, 2 of whom leave.

$$
p_{\mathrm{long}} = \dfrac{1}{5},
\qquad
\mathrm{odds} = \dfrac{1/5}{4/5} = \dfrac{1}{4},
\qquad
b_0 = \ln \dfrac{1}{4}.
$$

$$
p_{\mathrm{short}} = \dfrac{2}{5},
\qquad
\mathrm{odds} = \dfrac{2/5}{3/5} = \dfrac{2}{3},
\qquad
b_0 + b_1 = \ln \dfrac{2}{3}.
$$

$$
b_1 = \ln \dfrac{2}{3} - \ln \dfrac{1}{4} = \ln \dfrac{8}{3}.
$$

Then $\sigma(b_0) = 1/5$ and $\sigma(b_0+b_1) = 2/5$. Both sit below $1/2$, because $1 < 4$ and $2 < 3$. A classifier that flags a leave only when the probability exceeds $1/2$ calls every customer a stayer.

The same algebra on the fit rows, with one coefficient for month-to-month, uses the two fit rates. Month-to-month: 1348 leavers of 3110, which is $674/1555$. Everyone else: 156 leavers of 2516, which is $39/629$.

$$
\mathrm{odds}_{\mathrm{other}} = \dfrac{39}{590},
\qquad
\mathrm{odds}_{\mathrm{month}} = \dfrac{674}{881}.
$$

Both probabilities are below $1/2$, because $78 < 629$ and $1348 < 1555$. The cut at $1/2$ is again Stay-for-everyone. The two probabilities still rank the groups: $674/1555$ against $39/629$.

One year and two year are pooled in that single "other" rate. On the fit rows their own rates are $59/589$ and $19/669$.


In [1]:
# The five-and-five sketch. Both sigmoid values stay under one half.
from fractions import Fraction
from math import exp, log

def sigmoid(value):
    return 1 / (1 + exp(-value))

b0 = log(float(Fraction(1, 4)))
b1 = log(float(Fraction(8, 3)))
short = sigmoid(b0 + b1)
long = sigmoid(b0)
print(long, short)
assert abs(long - float(Fraction(1, 5))) < 1e-12
assert abs(short - float(Fraction(2, 5))) < 1e-12
assert long < 0.5 and short < 0.5
assert log(float(Fraction(2, 3))) - log(float(Fraction(1, 4))) == b1 or abs(
    (log(float(Fraction(2, 3))) - log(float(Fraction(1, 4)))) - b1
) < 1e-12


0.2 0.4


In [2]:
# Locate the telco table beside this notebook, or under the course root.
import csv
from pathlib import Path

def data_path():
    here = Path.cwd()
    name = "Telco-Customer-Churn.csv"
    for folder in [here, *here.parents]:
        nested = folder / "17-Customer Churn" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists():
            return direct
    raise FileNotFoundError(name)

def read_customers():
    with data_path().open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))
# Fit-set contract rates, then the two logistic probabilities.
import hashlib
from fractions import Fraction
from math import exp, log

rows = read_customers()

def held(customer_id):
    digest = hashlib.md5(customer_id.encode("utf-8")).hexdigest()
    return int(digest, 16) % 5 == 0

def sigmoid(value):
    return 1 / (1 + exp(-value))

fit = [row for row in rows if not held(row["customerID"])]
month = [row for row in fit if row["Contract"] == "Month-to-month"]
other = [row for row in fit if row["Contract"] != "Month-to-month"]
one = [row for row in fit if row["Contract"] == "One year"]
two = [row for row in fit if row["Contract"] == "Two year"]
month_left = sum(row["Churn"] == "Yes" for row in month)
other_left = sum(row["Churn"] == "Yes" for row in other)
assert (len(month), month_left) == (3110, 1348)
assert (len(other), other_left) == (2516, 156)
assert Fraction(month_left, len(month)) == Fraction(674, 1555)
assert Fraction(other_left, len(other)) == Fraction(39, 629)
assert Fraction(sum(row["Churn"] == "Yes" for row in one), len(one)) == Fraction(59, 589)
assert Fraction(sum(row["Churn"] == "Yes" for row in two), len(two)) == Fraction(19, 669)
b0 = log(float(Fraction(39, 590)))
b1 = log(float(Fraction(674, 881))) - b0
p_other = sigmoid(b0)
p_month = sigmoid(b0 + b1)
print(p_other, p_month)
assert abs(p_other - float(Fraction(39, 629))) < 1e-12
assert abs(p_month - float(Fraction(674, 1555))) < 1e-12
assert p_other < 0.5 and p_month < 0.5
assert 2 * 39 < 629 and 2 * 674 < 1555


0.06200317965023846 0.4334405144694534


## Pitfall

A logistic fit can look more elaborate than a sentence and still become Stay-for-everyone once you cut it at $1/2$. On this fit, $\sigma(b_0)$ and $\sigma(b_0+b_1)$ are the two group rates, and both rates are below one half. The path keeps the sentence from lesson 09, whose held-out $F_1$ is $488/861$, because that sentence flags the high-rate group on purpose.

## Summary

In the sketch, $b_0 = \ln(1/4)$ and $b_1 = \ln(8/3)$, and both probabilities are below $1/2$. On the fit rows the month-to-month probability is $674/1555$ and the other-contract probability is $39/629$. A cut at $1/2$ flags nobody.
